In [1]:
pip install pandas torch chronos-forecasting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip list

Package                                  Version
---------------------------------------- -------------------
a2a-sdk                                  0.3.25
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiobotocore                              3.3.0
aiofiles                                 22.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aioitertools                             0.13.0
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc               

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

import torch
from chronos import ChronosPipeline
CHRONOS_AVAILABLE = True

def seed_everything(seed=67):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if CHRONOS_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(67)

In [4]:
#score 90.39
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Попытка импортировать Chronos
try:
    import torch
    from chronos import ChronosPipeline
    CHRONOS_AVAILABLE = True
    print("OK Chronos is used")
except ImportError:
    CHRONOS_AVAILABLE = False
    print("Chronos недоступен, будет построена SOTA OOF-NLinear альтернатива.")

# 1. ЗАГРУЗКА ДАННЫХ
df = pd.read_csv('/kaggle/input/datasets/usefulwithgaming/gradient-rosta-kval/train_2.csv')
df['time_idx'] = df['Год'] * 12 + df['Месяц']
df = df.sort_values(['new_id', 'time_idx']).reset_index(drop=True)

# 2. ПОДГОТОВКА ТЕСТОВОГО ШАБЛОНА ДЛЯ МАРТА 2025
max_time_idx = df['time_idx'].max()
last_month_df = df[df['time_idx'] == max_time_idx].copy()

target_month = (last_month_df['Месяц'].iloc[0] % 12) + 1
target_year = last_month_df['Год'].iloc[0] + (1 if last_month_df['Месяц'].iloc[0] == 12 else 0)
target_time_idx = target_year * 12 + target_month

test_template = last_month_df.copy()
test_template['Месяц'] = target_month
test_template['Год'] = target_year
test_template['time_idx'] = target_time_idx
test_template['РТО'] = np.nan  # Будем предсказывать

df_full = pd.concat([df, test_template], ignore_index=True)
df_full = df_full.sort_values(['new_id', 'time_idx']).reset_index(drop=True)


# 3. ПОСТРОЕНИЕ ОПТИМИЗИРОВАННОГО КАЛЕНДАРЯ (Учитывает длину марта)
unique_dates = df_full[['Год', 'Месяц']].drop_duplicates().copy()

def calc_calendar(row):
    y, m = int(row['Год']), int(row['Месяц'])
    days = pd.Period(f"{y}-{m}").days_in_month
    start = pd.Timestamp(year=y, month=m, day=1)
    end = start + pd.offsets.MonthEnd(0)
    # Считаем будни (Mon-Fri) как прокси рабочих дней
    weekdays = sum(1 for d in pd.date_range(start, end) if d.weekday() < 5)
    return pd.Series([days, weekdays])

unique_dates[['days_in_month', 'weekdays_in_month']] = unique_dates.apply(calc_calendar, axis=1)
unique_dates['weekends_in_month'] = unique_dates['days_in_month'] - unique_dates['weekdays_in_month']

df_full = df_full.merge(unique_dates, on=['Год', 'Месяц'], how='left')


# 4. ПОДГОТОВКА И ОЧИСТКА ВРЕМЕННЫХ ТРАНЗАКЦИОННЫХ ПРИЗНАКОВ
time_dep_cols = [
    'Среднее количество промо товаров в чеке',
    'Среднее количество товаров в чеке',
    'Среднее количество отмен',
    'Трафик пеший, в час'
]

for col in time_dep_cols:
    if col in df_full.columns:
        if df_full[col].dtype == object:
            df_full[col] = df_full[col].astype(str).str.replace(',', '.').str.extract(r'(\d+\.?\d*)')[0].astype(float)
        
        # Заполнение пропусков медианой магазина с бэкапом на общую медиану
        store_medians = df_full.groupby('new_id')[col].transform('median')
        global_median = df_full[col].median()
        df_full[col] = df_full[col].fillna(store_medians).fillna(global_median if not pd.isna(global_median) else 0)


# Генерация лагов и SNaive-проекций для характеристик чека и трафика
for col in time_dep_cols:
    df_full[f'{col}_lag_1'] = df_full.groupby('new_id')[col].shift(1)
    df_full[f'{col}_lag_2'] = df_full.groupby('new_id')[col].shift(2)
    df_full[f'{col}_lag_12'] = df_full.groupby('new_id')[col].shift(12)
    df_full[f'{col}_lag_13'] = df_full.groupby('new_id')[col].shift(13)
    
    # Спроецированное значение показателя на текущий месяц (устраняет covariate shift)
    df_full[f'{col}_snaive_proj'] = df_full[f'{col}_lag_12'] * (df_full[f'{col}_lag_1'] / (df_full[f'{col}_lag_13'] + 1e-6))
    df_full[f'{col}_snaive_proj'] = df_full[f'{col}_snaive_proj'].fillna(df_full[f'{col}_lag_1'])
    
    # Динамика изменения
    df_full[f'{col}_mom_ratio'] = df_full[f'{col}_lag_1'] / (df_full[f'{col}_lag_2'] + 1e-6)
    df_full[f'{col}_yoy_ratio'] = df_full[f'{col}_lag_1'] / (df_full[f'{col}_lag_13'] + 1e-6)
    
    # Скользящее среднее
    df_full[f'{col}_roll_mean_3'] = df_full.groupby('new_id')[f'{col}_lag_1'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df_full[f'{col}_roll_mean_6'] = df_full.groupby('new_id')[f'{col}_lag_1'].transform(lambda x: x.rolling(6, min_periods=1).mean())

# Структурные соотношения чека
df_full['promo_to_total_ratio_lag_1'] = df_full['Среднее количество промо товаров в чеке_lag_1'] / (df_full['Среднее количество товаров в чеке_lag_1'] + 1e-6)
df_full['promo_to_total_ratio_proj'] = df_full['Среднее количество промо товаров в чеке_snaive_proj'] / (df_full['Среднее количество товаров в чеке_snaive_proj'] + 1e-6)

df_full['cancel_to_total_ratio_lag_1'] = df_full['Среднее количество отмен_lag_1'] / (df_full['Среднее количество товаров в чеке_lag_1'] + 1e-6)
df_full['cancel_to_total_ratio_proj'] = df_full['Среднее количество отмен_snaive_proj'] / (df_full['Среднее количество товаров в чеке_snaive_proj'] + 1e-6)


# 5. FEATURE ENGINEERING (Сбор базовых признаков)
def create_advanced_features(df):
    df = df.copy()
    
    # 5.1. Парсинг возраста магазина в месяцах
    try:
        df['open_datetime'] = pd.to_datetime(df['Дата открытия, категориальный'], errors='coerce')
        df['open_time_idx'] = df['open_datetime'].dt.year * 12 + df['open_datetime'].dt.month
        df['store_age_months'] = df['time_idx'] - df['open_time_idx']
        df['store_age_months'] = df['store_age_months'].fillna(-1)
    except Exception:
        df['store_age_months'] = -1

    # 5.2. Парсинг площади в числовой формат
    def parse_area(val):
        if pd.isna(val):
            return np.nan
        val_str = str(val).replace(',', '.')
        nums = [float(s) for s in re.findall(r'\d+\.?\d*', val_str)]
        if len(nums) == 2:
            return np.mean(nums)
        elif len(nums) == 1:
            return nums[0]
        return np.nan

    df['area_numeric'] = df['Торговая площадь, категориальный'].apply(parse_area)
    df['area_numeric'] = df['area_numeric'].fillna(df['area_numeric'].median() if not df['area_numeric'].isna().all() else -1)

    # 5.3. Безопасные лаги РТО
    for l in [1, 2, 3, 4, 5, 6, 11, 12, 13, 14]:
        df[f'lag_{l}'] = df.groupby('new_id')['РТО'].shift(l)
    
    # 5.4. ЦЕЛЕВАЯ: Логарифм прироста
    df['target_log_ratio'] = np.log((df['РТО'] + 1) / (df['lag_1'] + 1))
    
    # 5.5. Моментум и темпы роста
    df['momentum'] = (df['lag_1'] / (df['lag_2'] + 1e-6)) / (df['lag_2'] / (df['lag_3'] + 1e-6) + 1e-6)
    df['yoy_growth'] = df['lag_1'] / (df['lag_13'] + 1e-6)
    df['mom_growth'] = df['lag_1'] / (df['lag_2'] + 1e-6)
    df['season_trend_last_year'] = df['lag_12'] / (df['lag_13'] + 1e-6)
    
    # 5.6. Скользящие статистики на основе лагов
    df['rolling_mean_3'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['rolling_mean_6'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.rolling(6, min_periods=1).mean())
    df['rolling_mean_12'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.rolling(12, min_periods=1).mean())
    df['rolling_std_6'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.rolling(6, min_periods=1).std())
    
    # 5.7. Относительные скользящие метрики и волатильность
    df['rel_to_rolling_3'] = df['lag_1'] / (df['rolling_mean_3'] + 1e-6)
    df['rel_to_rolling_6'] = df['lag_1'] / (df['rolling_mean_6'] + 1e-6)
    df['rel_to_rolling_12'] = df['lag_1'] / (df['rolling_mean_12'] + 1e-6)
    df['volatility_ratio'] = (df['rolling_std_6'] + 1e-6) / (df['rolling_mean_6'] + 1e-6)
    
    # 5.8. Накопительный (expanding) профиль магазина
    df['store_mean_rto'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.expanding().mean())
    df['store_std_rto'] = df.groupby('new_id')['lag_1'].transform(lambda x: x.expanding().std()).fillna(0)

    # 5.9. Индикаторы эффективности работы
    df['rto_per_cashbox'] = df['lag_1'] / (df['Количество касс'] + 1e-6)
    df['traffic_efficiency'] = df['lag_1'] / (df['Трафик пеший, в час'] + 1e-6)
    
    # 5.10. Сезонность
    df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)
    
    return df

print("Вычисление базовых признаков...")
df_full = create_advanced_features(df_full)


# 6. ВНЕДРЕНИЕ СТРУКТУРЫ NLINEAR И ГЕНЕРАЦИЯ БЕЗОПАСНЫХ OOF-ПРЕДСКАЗАНИЙ ДЛЯ КАТБУСТА
# Вспомогательные классы нейросети
class MovingAvg(nn.Module):
    def __init__(self, kernel_size, stride):
        super(MovingAvg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        front = x[:, 0:1].repeat(1, (self.kernel_size - 1) // 2)
        end = x[:, -1:].repeat(1, self.kernel_size // 2)
        x_padded = torch.cat([front, x, end], dim=1).unsqueeze(1)
        return self.avg(x_padded).squeeze(1)

class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super(SeriesDecomp, self).__init__()
        self.moving_avg = MovingAvg(kernel_size, stride=1)

    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean

class NLinearModel(nn.Module):
    def __init__(self, seq_len, pred_len=1, kernel_size=5):
        super(NLinearModel, self).__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.decomposition = SeriesDecomp(kernel_size)
        self.Linear_Seasonal = nn.Linear(self.seq_len, self.pred_len)
        self.Linear_Trend = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x):
        seq_last = x[:, -1:].clone()
        seq_last = torch.clamp(seq_last, min=1e-5)
        x_norm = x / seq_last
        
        seasonal_init, trend_init = self.decomposition(x_norm)
        seasonal_output = self.Linear_Seasonal(seasonal_init)
        trend_output = self.Linear_Trend(trend_init)
        
        pred_norm = seasonal_output + trend_output
        return pred_norm * seq_last


# Расчет Сезонного наивного прогноза (SNaive) как признака
df_full['snaive_pred'] = df_full['lag_12'] * (df_full['lag_1'] / (df_full['lag_13'] + 1e-6))
df_full['snaive_pred'] = np.clip(df_full['snaive_pred'], 0, None)

df_full['chronos_pred'] = np.nan

if CHRONOS_AVAILABLE:
    try:
        print("\nИнициализация оригинальной модели Chronos-T5-Mini...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        
        pipeline = ChronosPipeline.from_pretrained(
            "amazon/chronos-t5-mini",
            device_map=device,
            torch_dtype=dtype,
        )
        
        sorted_times = sorted(df_full['time_idx'].unique())
        times_to_predict = sorted_times[-13:] 
        
        print(f"Запуск скользящей генерации Chronos для {len(times_to_predict)} месяцев...")
        for t in times_to_predict:
            hist_data = df_full[df_full['time_idx'] < t].copy()
            pivot_df = hist_data.pivot(index='new_id', columns='time_idx', values='РТО')
            store_ids = pivot_df.index.tolist()
            
            context_list = []
            for row in pivot_df.values:
                seq = row[~np.isnan(row)]
                if len(seq) > 24:
                    seq = seq[-24:]
                if len(seq) == 0:
                    seq = np.array([0.0])
                context_list.append(torch.tensor(seq, dtype=torch.float32))
            
            batch_size = 1024
            predictions = []
            for i in range(0, len(context_list), batch_size):
                batch_context = context_list[i:i+batch_size]
                with torch.no_grad():
                    batch_forecast = pipeline.predict(batch_context, prediction_length=1)
                median_forecasts = torch.median(batch_forecast, dim=1).values.cpu().numpy()
                predictions.extend(median_forecasts.flatten())
            
            pred_map = dict(zip(store_ids, np.clip(predictions, 0, None)))
            mask = df_full['time_idx'] == t
            df_full.loc[mask, 'chronos_pred'] = df_full.loc[mask, 'new_id'].map(pred_map)
            
        print("Мета-признаки Chronos успешно сгенерированы!")
    except Exception as e:
        print(f"Ошибка Chronos: {e}. Переключение на OOF-NLinear.")
        CHRONOS_AVAILABLE = False

if not CHRONOS_AVAILABLE:
    # Обучение NLinear по OOF-схеме на магазинах для избежания утечки данных!
    print("\n[SOTA] Запуск K-Fold генерации мета-признаков NLinear...")
    
    # 2D матрица РТО истории (new_id x time_idx)
    pivot_df = df_full[df_full['time_idx'] < target_time_idx].pivot(index='new_id', columns='time_idx', values='РТО')
    pivot_df = pivot_df.ffill(axis=1).bfill(axis=1).fillna(0.0)
    
    H = 18  # Окно контекста
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    unique_ids = pivot_df.index.values
    
    # 5-фолдовый разбиение магазинов для создания мета-предсказаний
    kf_meta = KFold(n_splits=5, shuffle=True, random_state=100)
    
    for fold_meta, (tr_ids, val_ids) in enumerate(kf_meta.split(unique_ids)):
        print(f"Обучение NLinear мета-модели на Fold {fold_meta}...")
        shops_tr = unique_ids[tr_ids]
        shops_val = unique_ids[val_ids]
        
        pivot_tr = pivot_df.loc[shops_tr]
        
        X_tr_list, Y_tr_list = [], []
        for t in range(H, len(pivot_tr.columns)):
            X_tr_list.append(pivot_tr.iloc[:, t-H:t].values)
            Y_tr_list.append(pivot_tr.iloc[:, t:t+1].values)
            
        X_tr_np = np.concatenate(X_tr_list, axis=0)
        Y_tr_np = np.concatenate(Y_tr_list, axis=0)
        
        tr_tensor_x = torch.tensor(X_tr_np, dtype=torch.float32)
        tr_tensor_y = torch.tensor(Y_tr_np, dtype=torch.float32)
        meta_dataset = TensorDataset(tr_tensor_x, tr_tensor_y)
        meta_loader = DataLoader(meta_dataset, batch_size=512, shuffle=True)
        
        model_meta = NLinearModel(seq_len=H, pred_len=1, kernel_size=5).to(device)
        criterion = nn.L1Loss()
        opt = torch.optim.Adam(model_meta.parameters(), lr=0.005, weight_decay=1e-5)
        
        model_meta.train()
        for epoch in range(80):
            for bx, by in meta_loader:
                bx, by = bx.to(device), by.to(device)
                pred = model_meta(bx)
                loss = criterion(pred, by)
                opt.zero_grad()
                loss.backward()
                opt.step()
                
        # Предсказания для валидационных магазинов фолда на всей истории
        model_meta.eval()
        pivot_val = pivot_df.loc[shops_val]
        with torch.no_grad():
            for t in range(H, len(df_full['time_idx'].unique())):
                time_val = sorted(df_full['time_idx'].unique())[t]
                
                if time_val == target_time_idx:
                    context = pivot_val.iloc[:, -H:].values
                else:
                    col_idx = pivot_val.columns.get_loc(time_val)
                    context = pivot_val.iloc[:, col_idx-H:col_idx].values
                    
                context_tensor = torch.tensor(context, dtype=torch.float32).to(device)
                preds = model_meta(context_tensor).cpu().numpy().flatten()
                preds = np.clip(preds, 0, None)
                
                pred_map = dict(zip(shops_val, preds))
                mask = (df_full['time_idx'] == time_val) & (df_full['new_id'].isin(shops_val))
                df_full.loc[mask, 'chronos_pred'] = df_full.loc[mask, 'new_id'].map(pred_map)
                
    print("OOF NLinear мета-признаки успешно сгенерированы!")

# Заполняем редкие пропуски в ранней истории SNaive
df_full['chronos_pred'] = df_full['chronos_pred'].fillna(df_full['snaive_pred'])

# Формируем мета-фичи для CatBoost
df_full['log_chronos_pred'] = np.log1p(df_full['chronos_pred'])
df_full['log_snaive_pred'] = np.log1p(df_full['snaive_pred'])

df_full['chronos_log_ratio'] = np.log((df_full['chronos_pred'] + 1) / (df_full['lag_1'] + 1))
df_full['snaive_log_ratio'] = np.log((df_full['snaive_pred'] + 1) / (df_full['lag_1'] + 1))


# 7. НАСТРОЙКА КАТЕГОРИЙ И ИТОГОВОГО СПИСКА ПРИЗНАКОВ
categorical_features = [
    'Регион', 'Флаг алкогольной лицензии'
]
for col in categorical_features:
    df_full[col] = df_full[col].astype(str).fillna('NA')

features = [
    'Месяц', 'month_sin', 'month_cos', 'store_age_months', 'area_numeric',
    'days_in_month', 'weekdays_in_month', 'weekends_in_month',
    'yoy_growth', 'mom_growth', 'season_trend_last_year', 'momentum', 
    'rel_to_rolling_3', 'rel_to_rolling_6', 'rel_to_rolling_12', 'volatility_ratio',
    'store_mean_rto', 'store_std_rto',
    'log_chronos_pred', 'log_snaive_pred',     # Хронос (или OOF NLinear) и SNaive в лог-масштабе
    'chronos_log_ratio', 'snaive_log_ratio',   # Хронос (или OOF NLinear) и SNaive в приростах
    'Численность населения', 'Количество касс', 'rto_per_cashbox', 'traffic_efficiency',
    'Среднее количество промо товаров в чеке', 'Рабочие часы в день',
    'Среднее количество товаров в чеке', 'Среднее количество отмен',
    
    # Новые продвинутые транзакционные и структурные переменные
    'Среднее количество промо товаров в чеке_lag_1', 'Среднее количество промо товаров в чеке_lag_2', 'Среднее количество промо товаров в чеке_snaive_proj', 'Среднее количество промо товаров в чеке_mom_ratio', 'Среднее количество промо товаров в чеке_yoy_ratio', 'Среднее количество промо товаров в чеке_roll_mean_3', 'Среднее количество промо товаров в чеке_roll_mean_6',
    'Среднее количество товаров в чеке_lag_1', 'Среднее количество товаров в чеке_lag_2', 'Среднее количество товаров в чеке_snaive_proj', 'Среднее количество товаров в чеке_mom_ratio', 'Среднее количество товаров в чеке_yoy_ratio', 'Среднее количество товаров в чеке_roll_mean_3', 'Среднее количество товаров в чеке_roll_mean_6',
    'Среднее количество отмен_lag_1', 'Среднее количество отмен_lag_2', 'Среднее количество отмен_snaive_proj', 'Среднее количество отмен_mom_ratio', 'Среднее количество отмен_yoy_ratio', 'Среднее количество отмен_roll_mean_3', 'Среднее количество отмен_roll_mean_6',
    
    'Трафик пеший, в час_lag_1', 'Трафик пеший, в час_lag_2', 'Трафик пеший, в час_snaive_proj',
    
    'promo_to_total_ratio_lag_1', 'promo_to_total_ratio_proj',
    'cancel_to_total_ratio_lag_1', 'cancel_to_total_ratio_proj'
] + categorical_features


# 8. РАЗДЕЛЕНИЕ НА TRAIN И TEST (С экспоненциальным весом времени)
train_full = df_full[df_full['target_log_ratio'].notna() & df_full['lag_13'].notna()].copy()
test_df = df_full[df_full['time_idx'] == target_time_idx].copy()

# Добавление весов времени
train_full['sample_weight'] = np.exp((train_full['time_idx'] - train_full['time_idx'].min()) / 12)


# 9. КРОСС-ВАЛИДАЦИЯ С ТАРГЕТ-КОДИРОВАНИЕМ И SEED-ENSEMBLING (CatBoost)
print("\nЗапуск итоговой кросс-валидации модели CatBoost с семенами (Seeds) для победы...")
unique_shops = train_full['new_id'].unique()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

seeds = [42]
test_preds_total = []

for seed in seeds:
    print(f"\n==== ОБУЧЕНИЕ ДЛЯ SEED {seed} ====")
    for fold, (train_idx, val_idx) in enumerate(kf.split(unique_shops)):
        print(f"--- Seed {seed} | Fold {fold} ---")
        shops_train = unique_shops[train_idx]
        shops_val = unique_shops[val_idx]
        
        d_train = train_full[train_full['new_id'].isin(shops_train)].copy()
        d_val = train_full[train_full['new_id'].isin(shops_val)].copy()
        
        # --- БЕЗОПАСНОЕ ТАРГЕТ КОДИРОВАНИЕ ---
        reg_map = d_train.groupby('Регион')['target_log_ratio'].mean().to_dict()
        d_train['reg_target_enc'] = d_train['Регион'].map(reg_map)
        d_val['reg_target_enc'] = d_val['Регион'].map(reg_map).fillna(d_train['target_log_ratio'].mean())
        
        city_map = d_train.groupby('Населенный пункт')['target_log_ratio'].mean().to_dict()
        d_train['city_target_enc'] = d_train['Населенный пункт'].map(city_map)
        d_val['city_target_enc'] = d_val['Населенный пункт'].map(city_map).fillna(d_train['target_log_ratio'].mean())
        
        test_df_fold = test_df.copy()
        test_df_fold['reg_target_enc'] = test_df_fold['Регион'].map(reg_map).fillna(d_train['target_log_ratio'].mean())
        test_df_fold['city_target_enc'] = test_df_fold['Населенный пункт'].map(city_map).fillna(d_train['target_log_ratio'].mean())
        
        current_features = features + ['reg_target_enc', 'city_target_enc']
        
        model = CatBoostRegressor(
            iterations=6000,          # Увеличено до 6000 для более плавной сходимости
            learning_rate=0.012,       # Шаг обучения уменьшен до 0.012 для стабильности
            depth=8,                  # Оптимальная глубина
            l2_leaf_reg=15,
            bootstrap_type='Bernoulli',
            subsample=0.8,
            task_type="GPU" if torch.cuda.is_available() else "CPU",
            loss_function='MAE',      # Оптимизация MAE под MAPE
            eval_metric='MAPE',
            random_seed=seed + fold,
            verbose=500
        )
        
        model.fit(
            d_train[current_features], d_train['target_log_ratio'],
            sample_weight=d_train['sample_weight'],
            cat_features=categorical_features,
            eval_set=(d_val[current_features], d_val['target_log_ratio']),
            early_stopping_rounds=350
        )
        
        pred_log_ratio = model.predict(test_df_fold[current_features])
        pred_log_ratio = np.clip(pred_log_ratio, -2.5, 2.5)
        
        # Обратная трансформация предсказания
        fold_pred_rto = (test_df_fold['lag_1'].values + 1) * np.exp(pred_log_ratio) - 1
        fold_pred_rto = np.clip(fold_pred_rto, 0, None)
        
        test_preds_total.append(fold_pred_rto)

# 10. ФОРМИРОВАНИЕ ИТОГОВОГО СУБМИШЕНА С ВЫРАВНИВАНИЕМ (18657 СТРОК)
final_rto = np.mean(test_preds_total, axis=0)

submission = pd.DataFrame({'new_id': test_df['new_id'], 'rto': final_rto})

# Выравнивание с официальным шаблоном
test_path = '/kaggle/input/datasets/usefulwithgaming/gradient-rosta-kval/test.csv'
if os.path.exists(test_path):
    print("\nНайден официальный test.csv. Синхронизируем строки...")
    official_test = pd.read_csv(test_path)
    submission = official_test[['new_id']].merge(submission, on='new_id', how='left')
    submission['rto'] = submission['rto'].fillna(submission['rto'].median())


print(f"\nРезультирующее число строк в файле: {len(submission)}")
print(f"Средний прогнозируемый РТО: {submission['rto'].mean():.2f}")
print("Финальный SOTA файл test.csv готов к отправке!")

OK Chronos is used
Вычисление базовых признаков...

Инициализация оригинальной модели Chronos-T5-Mini...
Запуск скользящей генерации Chronos для 13 месяцев...
Мета-признаки Chronos успешно сгенерированы!

Запуск итоговой кросс-валидации модели CatBoost с семенами (Seeds) для победы...

==== ОБУЧЕНИЕ ДЛЯ SEED 42 ====
--- Seed 42 | Fold 0 ---


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.0774085	test: 0.0735245	best: 0.0735245 (0)	total: 116ms	remaining: 11m 37s
500:	learn: 0.0302968	test: 0.0329087	best: 0.0329087 (500)	total: 10.1s	remaining: 1m 50s
1000:	learn: 0.0275044	test: 0.0321723	best: 0.0321720 (999)	total: 19.5s	remaining: 1m 37s
1500:	learn: 0.0252036	test: 0.0319771	best: 0.0319746 (1492)	total: 29.3s	remaining: 1m 27s
2000:	learn: 0.0234678	test: 0.0319546	best: 0.0319474 (1732)	total: 39.1s	remaining: 1m 18s
bestTest = 0.0319473678
bestIteration = 1732
Shrink model to first 1733 iterations.
--- Seed 42 | Fold 1 ---


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.0775339	test: 0.0733146	best: 0.0733146 (0)	total: 19.5ms	remaining: 1m 57s
500:	learn: 0.0303297	test: 0.0327938	best: 0.0327938 (500)	total: 9.45s	remaining: 1m 43s
1000:	learn: 0.0275595	test: 0.0320178	best: 0.0320178 (1000)	total: 18.8s	remaining: 1m 33s
1500:	learn: 0.0252863	test: 0.0318162	best: 0.0318158 (1490)	total: 28.4s	remaining: 1m 25s
2000:	learn: 0.0235666	test: 0.0317865	best: 0.0317790 (1791)	total: 38.2s	remaining: 1m 16s
2500:	learn: 0.0222221	test: 0.0317718	best: 0.0317613 (2408)	total: 48.2s	remaining: 1m 7s
bestTest = 0.03176133864
bestIteration = 2408
Shrink model to first 2409 iterations.
--- Seed 42 | Fold 2 ---


...

0:	learn: 0.0770743	test: 0.0748999	best: 0.0748999 (0)	total: 28.9ms	remaining: 2m 53s
500:	learn: 0.0301252	test: 0.0336079	best: 0.0336079 (500)	total: 9.88s	remaining: 1m 48s
1000:	learn: 0.0273421	test: 0.0327949	best: 0.0327949 (1000)	total: 19.5s	remaining: 1m 37s
1500:	learn: 0.0250294	test: 0.0325657	best: 0.0325649 (1495)	total: 29.2s	remaining: 1m 27s
2000:	learn: 0.0233159	test: 0.0325394	best: 0.0325201 (1757)	total: 38.9s	remaining: 1m 17s
bestTest = 0.03252014735
bestIteration = 1757
Shrink model to first 1758 iterations.
--- Seed 42 | Fold 3 ---


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.0774048	test: 0.0736880	best: 0.0736880 (0)	total: 29.9ms	remaining: 2m 59s
500:	learn: 0.0303127	test: 0.0330975	best: 0.0330975 (500)	total: 9.57s	remaining: 1m 45s
1000:	learn: 0.0274936	test: 0.0323749	best: 0.0323749 (1000)	total: 19s	remaining: 1m 34s
1500:	learn: 0.0251984	test: 0.0322036	best: 0.0322029 (1477)	total: 28.7s	remaining: 1m 26s
2000:	learn: 0.0235008	test: 0.0322058	best: 0.0321773 (1675)	total: 38.4s	remaining: 1m 16s
bestTest = 0.03217730703
bestIteration = 1675
Shrink model to first 1676 iterations.
--- Seed 42 | Fold 4 ---


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.0774177	test: 0.0736963	best: 0.0736963 (0)	total: 19.1ms	remaining: 1m 54s
500:	learn: 0.0303644	test: 0.0328377	best: 0.0328377 (500)	total: 9.85s	remaining: 1m 48s
1000:	learn: 0.0275286	test: 0.0320340	best: 0.0320340 (1000)	total: 19.6s	remaining: 1m 38s
1500:	learn: 0.0252690	test: 0.0318253	best: 0.0318251 (1499)	total: 29.7s	remaining: 1m 28s
2000:	learn: 0.0235134	test: 0.0318002	best: 0.0317922 (1894)	total: 39.7s	remaining: 1m 19s
2500:	learn: 0.0221299	test: 0.0317899	best: 0.0317859 (2259)	total: 49.5s	remaining: 1m 9s
bestTest = 0.03178472802
bestIteration = 2529
Shrink model to first 2530 iterations.

Результирующее число строк в файле: 18657
Средний прогнозируемый РТО: 103139891.62
Финальный SOTA файл test.csv готов к отправке!


In [6]:
#score 90.45 APE = 100 + предсказания 90.39
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

# ==========================================
# НАСТРОЙКА ПУТЕЙ И ПОРОГОВ (Укажите ваши файлы)
# ==========================================
TRAIN_PATH = '/kaggle/input/datasets/usefulwithgaming/gradient-rosta-kval/train_2.csv'
#SUB_INPUT_PATH = '/kaggle/input/datasets/usefulwithgaming/dataset-for-blending/_cb_ts_10.csv'   # Ваша старая посылка без учета выбросов
SUB_OUTPUT_PATH = 'test.csv'                 # Новая скорректированная посылка
APE_THRESHOLD = 100                        # Порог аномальности в % (20, 50 или 100)


# 1. ЗАГРУЗКА И ПОДГОТОВКА ИСТОРИИ
print("Загрузка исторических данных...")
df = pd.read_csv(TRAIN_PATH)
df['time_idx'] = df['Год'] * 12 + df['Месяц']
df = df.sort_values(['new_id', 'time_idx']).reset_index(drop=True)

# 2. МАТЕМАТИЧЕСКИЙ РАСЧЕТ РОБАСТНОГО ПРОГНОЗА ДЛЯ МАРТА 2025 (БЕЗ ОБУЧЕНИЯ)
print("Расчет робастных мартовских прогнозов из истории...")
pivot_df = df.pivot(index='new_id', columns='time_idx', values='РТО')

# Опеределяем временные индексы относительно марта 2025
idx_jan_25 = 2025 * 12 + 1   # lag_2 относительно марта
idx_dec_24 = 2024 * 12 + 12  # lag_3 относительно марта
idx_nov_24 = 2024 * 12 + 11  # lag_4 относительно марта
idx_mar_24 = 2024 * 12 + 3   # lag_12 относительно марта
idx_jan_24 = 2024 * 12 + 1   # lag_14 относительно марта

# Извлекаем чистые исторические векторы продаж
rto_jan_25 = pivot_df[idx_jan_25]
rto_dec_24 = pivot_df[idx_dec_24]
rto_nov_24 = pivot_df[idx_nov_24]
rto_mar_24 = pivot_df[idx_mar_24]
rto_jan_24 = pivot_df[idx_jan_24]
rto_mean = pivot_df.mean(axis=1)

# Вычисляем робастный SNaive и скользящее среднее
robust_snaive = rto_mar_24 * (rto_jan_25 / (rto_jan_24 + 1e-6))
robust_rolling = (rto_jan_25 + rto_dec_24 + rto_nov_24) / 3.0

# 70% SNaive + 30% скользящий профиль
fallback_preds = 0.7 * robust_snaive + 0.3 * robust_rolling

# Подстраховка пропусков средними значениями магазина
fallback_preds = fallback_preds.fillna(robust_rolling).fillna(rto_mean).fillna(0)
fallback_preds = np.clip(fallback_preds, 0, None)
fallback_dict = fallback_preds.to_dict()


# 3. ПОЛУЧЕНИЕ СПИСКА АНОМАЛИЙ ЧЕРЕЗ БЫСТРУЮ ОЦЕНКУ ФЕВРАЛЯ
print("\nЗапуск легкого диагностического CatBoost для выявления аномалий на феврале...")
feb_2025_time_idx = 2025 * 12 + 2

# Генерируем минимальный набор лагов только для оценки февраля
for l in [1, 2, 3, 12, 13]:
    df[f'lag_{l}'] = df.groupby('new_id')['РТО'].shift(l)

df['target_log_ratio'] = np.log((df['РТО'] + 1) / (df['lag_1'] + 1))
df['snaive_pred'] = df['lag_12'] * (df['lag_1'] / (df['lag_13'] + 1e-6))
df['snaive_pred'] = np.clip(df['snaive_pred'], 0, None)
df['log_snaive_pred'] = np.log1p(df['snaive_pred'])
df['snaive_log_ratio'] = np.log((df['snaive_pred'] + 1) / (df['lag_1'] + 1))
df['yoy_growth'] = df['lag_1'] / (df['lag_13'] + 1e-6)
df['mom_growth'] = df['lag_1'] / (df['lag_2'] + 1e-6)
df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)

categorical_features = ['Регион', 'Флаг алкогольной лицензии']
for col in categorical_features:
    df[col] = df[col].astype(str).fillna('NA')

features_light = [
    'Месяц', 'month_sin', 'month_cos', 'yoy_growth', 'mom_growth',
    'log_snaive_pred', 'snaive_log_ratio', 'lag_1', 'lag_2', 'lag_3'
] + categorical_features

train_feb = df[(df['time_idx'] < feb_2025_time_idx) & df['target_log_ratio'].notna() & df['lag_13'].notna()].copy()
val_feb = df[df['time_idx'] == feb_2025_time_idx].copy()
train_feb['sample_weight'] = np.exp((train_feb['time_idx'] - train_feb['time_idx'].min()) / 12)

unique_shops = train_feb['new_id'].unique()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

feb_preds = np.zeros(len(val_feb))

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_shops)):
    shops_train = unique_shops[train_idx]
    d_train = train_feb[train_feb['new_id'].isin(shops_train)].copy()
    
    # Сглаженный таргет-энкодинг
    reg_map = d_train.groupby('Регион')['target_log_ratio'].mean().to_dict()
    city_map = d_train.groupby('Населенный пункт')['target_log_ratio'].mean().to_dict()
    d_train['reg_target_enc'] = d_train['Регион'].map(reg_map)
    d_train['city_target_enc'] = d_train['Населенный пункт'].map(city_map)
    
    val_feb_fold = val_feb.copy()
    val_feb_fold['reg_target_enc'] = val_feb_fold['Регион'].map(reg_map).fillna(d_train['target_log_ratio'].mean())
    val_feb_fold['city_target_enc'] = val_feb_fold['Населенный пункт'].map(city_map).fillna(d_train['target_log_ratio'].mean())
    
    curr_feats = features_light + ['reg_target_enc', 'city_target_enc']
    
    # Быстрый и легкий CatBoost
    model_cv = CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function='MAE',
        random_seed=42 + fold,
        verbose=False,
        task_type='GPU'
    )
    model_cv.fit(d_train[curr_feats], d_train['target_log_ratio'], sample_weight=d_train['sample_weight'], cat_features=categorical_features)
    feb_preds += model_cv.predict(val_feb_fold[curr_feats]) / 5.0

val_feb['pred_log_ratio'] = feb_preds
val_feb['pred_rto'] = (val_feb['lag_1'].values + 1) * np.exp(val_feb['pred_log_ratio']) - 1
val_feb['pred_rto'] = np.clip(val_feb['pred_rto'], 0, None)
val_feb['APE'] = np.abs(val_feb['РТО'] - val_feb['pred_rto']) / (val_feb['РТО'] + 1e-6) * 100

# Вытаскиваем только те id, где APE превышает заданный порог
outlier_ids = set(val_feb[val_feb['APE'] > APE_THRESHOLD]['new_id'].unique())
print(f"Обнаружено {len(outlier_ids)} магазинов с APE > {APE_THRESHOLD}% на валидации февраля.")




sub_df = submission.copy()

# Проводим замену
replaced_counter = 0
for idx, row in sub_df.iterrows():
    shop_id = row['new_id']
    if shop_id in outlier_ids:
        # Берем подготовленный робастный мартовский прогноз для этого id
        sub_df.at[idx, 'rto'] = fallback_dict.get(shop_id, row['rto'])
        replaced_counter += 1

# Сохраняем финальный откорректированный файл
sub_df.to_csv(SUB_OUTPUT_PATH, index=False)

print(f"\nКорректировка успешно завершена!")
print(f"Заменено предсказаний: {replaced_counter} из {len(sub_df)}")
print(f"Финальный скорректированный файл сохранен как '{SUB_OUTPUT_PATH}' и готов к отправке в систему!")

Загрузка исторических данных...
Расчет робастных мартовских прогнозов из истории...

Запуск легкого диагностического CatBoost для выявления аномалий на феврале...


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Обнаружено 66 магазинов с APE > 100% на валидации февраля.

Корректировка успешно завершена!
Заменено предсказаний: 66 из 18657
Финальный скорректированный файл сохранен как 'test.csv' и готов к отправке в систему!
